# YOLO26n detectors — Colab / Kaggle GPU training

Retrains the pipeline's three small detectors on **YOLO26n** (Ultralytics,
Sept 2025): NMS-free end-to-end head, ~43% faster CPU inference than YOLOv8n,
*higher* mAP. This box is CPU-only — training must run on a free GPU (Colab
T4 / Kaggle P100). Inference stays on the Azure box's onnxruntime CPU.

| detector | file it replaces | classes | input |
|---|---|---|---|
| **panel+text** | `models/manga-panel-yolo/manga_panel_detector_fp32_1024.onnx` | panel, text | 1024 |
| **face** | `models/anime-face/face_yolo26n.onnx` *(new name — auto-picked over `face_v1.4_n.onnx`)* | face | 640 |
| **bubble** | `models/comic-bubble/comic-speech-bubble-detector-yolo26n.onnx` *(new name — auto-picked over the `.pt`)* | bubble | 1024 |

`master_pipeline.py` already resolves each detector from an ordered
candidate list (`_resolve_model`, `FACE_ONNX_CANDIDATES`,
`BUBBLE_MODEL_CANDIDATES`) and `_detect_faces_raw` parses **both** the
classic v8 `(4+nc, N)` output and the YOLO26 end-to-end `(N, 6)` output —
so a correct export is a genuine drop-in, no code edit.

Runtime: **GPU**. ~1–2 h per detector. Do the panel one first; it's the
highest-value.


In [ ]:
!pip -q install 'ultralytics>=8.4.142' onnx onnxslim onnxruntime
import torch, ultralytics
print('ultralytics', ultralytics.__version__)
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
# YOLO26 weights auto-download from Ultralytics assets on first use.


## 1. Panel + text detector

Dataset: a YOLO folder (`images/{train,val}`, `labels/{train,val}`,
`data.yaml`) from `pipeline/training/build_dataset.py` — see that script and
`pipeline/training/README.md` for the webtoon data sources (Roboflow
"Webtoon Panel" 2.8k imgs + `autolabel_bootstrap.py` on our own chapters +
recap-frame alignment). `data.yaml` names must be `['panel', 'text']` in
that order (class 0 = panel, class 1 = text) to match `_yolo_detect_page`.


In [ ]:
# --- get the dataset ---
# Option A: upload panel-yolo.zip via the Files panel, then:
!unzip -q -o panel-yolo.zip -d /content/data
DATA = '/content/data/webtoon-yolo/data.yaml'
# Option B (Colab + Drive):
# from google.colab import drive; drive.mount('/content/drive')
# DATA = '/content/drive/MyDrive/webtoon-yolo/data.yaml'
import yaml; print(yaml.safe_load(open(DATA)))


In [ ]:
from ultralytics import YOLO

# Fine-tune, not scratch. yolo26n.pt is the COCO-pretrained nano checkpoint.
model = YOLO('yolo26n.pt')

model.train(
    data=DATA,
    epochs=120,
    imgsz=1024,                 # webtoon pages are tall — keep resolution up
    batch=16,
    rect=False,
    mosaic=0.5, mixup=0.0, copy_paste=0.0,
    degrees=0.0, shear=0.0, perspective=0.0,   # comics aren't rotated
    fliplr=0.0,                 # reading order / speech-tail direction matters
    hsv_h=0.0, hsv_s=0.3, hsv_v=0.3,
    patience=25,
    project='/content/runs', name='panel_y26',
)


In [ ]:
# --- eval + export (panel) ---
best = '/content/runs/panel_y26/weights/best.pt'
m = YOLO(best)
print(m.val(data=DATA, imgsz=1024).box.map, 'mAP50-95')

# nms=True bakes the end-to-end decode in -> ONNX output (1, 300, 6) =
# [x1, y1, x2, y2, score, cls] in letterbox px, exactly what
# _yolo_detect_page already expects.
onnx_path = m.export(format='onnx', imgsz=1024, opset=13, simplify=True, nms=True)
print('exported:', onnx_path)
from google.colab import files; files.download(onnx_path)   # -> rename on deploy


## 2. Face detector

Replaces `models/anime-face/face_v1.4_n.onnx`. Manhwa faces are painted
semi-realistically, so an anime-only face set under-fires — mix in manhwa
faces. Data options:

* **`autolabel_bootstrap.py`** already writes face boxes (it runs the current
  detector); correct the misses in Label Studio / CVAT.
* Public anime-face sets (Roboflow `anime-face-detection`, `manga-face`) as a base.
* Recap frames: a recap channel crops tight to a character's face constantly.

Single class `['face']`. Input 640 (bands are ~2:1, `_detect_regions_tallsafe`
splits tall strips).


In [ ]:
!unzip -q -o face-yolo.zip -d /content/data
DATA_FACE = '/content/data/face-yolo/data.yaml'
import yaml; print(yaml.safe_load(open(DATA_FACE)))


In [ ]:
from ultralytics import YOLO
model = YOLO('yolo26n.pt')
model.train(
    data=DATA_FACE,
    epochs=100, imgsz=640, batch=32, rect=False,
    mosaic=0.5, degrees=0.0, shear=0.0, perspective=0.0, fliplr=0.5,
    hsv_h=0.0, hsv_s=0.3, hsv_v=0.4, patience=20,
    project='/content/runs', name='face_y26',
)
m = YOLO('/content/runs/face_y26/weights/best.pt')
print(m.val(data=DATA_FACE, imgsz=640).box.map, 'mAP50-95')
p = m.export(format='onnx', imgsz=640, opset=13, simplify=True, nms=True)
from google.colab import files; files.download(p)


## 3. Speech-bubble detector

Replaces `models/comic-bubble/comic-speech-bubble-detector.pt` (YOLOv8m).
`YOLO()` loads the `.onnx` the same way, so this is a drop-in and faster.

Data: **`ogkalu/comic-text-and-bubble-detector`** (RT-DETR, already vendored
under `models/comic-text-and-bubble-detector/`) can auto-label a big pile of
our chapters; `Kiuyha/Manga-Bubble-YOLO` and `kitsumed/yolov8m_seg-speech-bubble`
ship label sets too. Single class `['bubble']`, input 1024.

**Optional but valuable:** add a `tail` class (the bubble's pointer) — it's
the key signal for #9 speaker attribution. Manga109Dialog / PopManga have
tail annotations. If you train `['bubble', 'tail']`, set
`RECAP_BUBBLE_TAIL_CLASS=1` and the speaker module picks it up.


In [ ]:
!unzip -q -o bubble-yolo.zip -d /content/data
DATA_BUBBLE = '/content/data/bubble-yolo/data.yaml'
import yaml; print(yaml.safe_load(open(DATA_BUBBLE)))


In [ ]:
from ultralytics import YOLO
model = YOLO('yolo26n.pt')
model.train(
    data=DATA_BUBBLE,
    epochs=100, imgsz=1024, batch=16, rect=False,
    mosaic=0.3, degrees=0.0, shear=0.0, perspective=0.0, fliplr=0.0,
    hsv_h=0.0, hsv_s=0.2, hsv_v=0.3, patience=20,
    project='/content/runs', name='bubble_y26',
)
m = YOLO('/content/runs/bubble_y26/weights/best.pt')
print(m.val(data=DATA_BUBBLE, imgsz=1024).box.map, 'mAP50-95')
p = m.export(format='onnx', imgsz=1024, opset=13, simplify=True, nms=True)
from google.colab import files; files.download(p)


## 4. Deploy to the Azure box

```bash
# panel (keep the existing filename — it's already the primary path)
cp panel_y26.onnx  pipeline/models/manga-panel-yolo/manga_panel_detector_fp32_1024.onnx

# face — NEW filename; _resolve_model prefers it over face_v1.4_n.onnx
cp face_y26.onnx   pipeline/models/anime-face/face_yolo26n.onnx

# bubble — NEW filename; preferred over the .pt
cp bubble_y26.onnx pipeline/models/comic-bubble/comic-speech-bubble-detector-yolo26n.onnx

# verify nothing regressed
python -m pytest tests/test_image_slicing.py -q
python pipeline/master_pipeline.py --input <a-chapter-dir> --output /tmp/y26 --dry-run-slices
```

Rollback is just deleting the new file — the candidate list falls back to the
shipped weight. `RECAP_FACE_MODEL` / `RECAP_BUBBLE_MODEL` env vars force a
specific path if you want to A/B without moving files.

### Sanity-check the export output shape

```python
import onnxruntime as ort, numpy as np
s = ort.InferenceSession('face_y26.onnx')
o = s.run(None, {s.get_inputs()[0].name: np.zeros((1,3,640,640), np.float32)})[0]
print(o.shape)   # expect (1, N, 6); _detect_faces_raw handles (N,6) and (4+nc,N)
```
